In [0]:


CREATE OR REPLACE TABLE telecom_lakehouse.bronze.erp_gl_raw
USING DELTA
COMMENT 'Synthetic ERP general ledger entries using double-entry accounting'
AS

WITH invoice_ranked AS (
    SELECT
        invoice_id,
        invoice_date,
        invoice_amount,
        CEIL(
            ROW_NUMBER() OVER (ORDER BY invoice_id) / 2.0
        ) AS batch_number
    FROM telecom_lakehouse.bronze.billing_invoices_raw
),

invoice_batches AS (
    SELECT
        CONCAT('INV_JRN_', LPAD(CAST(batch_number AS STRING), 8, '0')) AS journal_id,
        MIN(invoice_date) AS posting_date,
        SUM(invoice_amount) AS journal_amount,
        'INVOICE' AS source_type,
        CONCAT('INV_BATCH_', CAST(batch_number AS STRING)) AS source_reference
    FROM invoice_ranked
    GROUP BY batch_number
),

payment_ranked AS (
    SELECT
        payment_id,
        payment_date,
        payment_amount,
        NTILE(25000) OVER (ORDER BY payment_id) AS batch_number
    FROM telecom_lakehouse.bronze.customer_payments_raw
),

payment_batches AS (
    SELECT
        CONCAT('PAY_JRN_', LPAD(CAST(batch_number AS STRING), 8, '0')) AS journal_id,
        MIN(payment_date) AS posting_date,
        SUM(payment_amount) AS journal_amount,
        'PAYMENT' AS source_type,
        CONCAT('PAY_BATCH_', CAST(batch_number AS STRING)) AS source_reference
    FROM payment_ranked
    GROUP BY batch_number
),

journal_headers AS (
    SELECT * FROM invoice_batches
    UNION ALL
    SELECT * FROM payment_batches
),

journal_lines AS (

    -- Invoice: Debit Accounts Receivable
    SELECT
        journal_id,
        posting_date,
        '1100' AS account_code,
        'ACCOUNTS_RECEIVABLE' AS account_name,
        journal_amount AS debit_amount,
        CAST(0 AS DECIMAL(18,2)) AS credit_amount,
        source_type,
        source_reference
    FROM journal_headers
    WHERE source_type = 'INVOICE'

    UNION ALL

    -- Invoice: Credit Telecom Revenue
    SELECT
        journal_id,
        posting_date,
        '4100' AS account_code,
        'TELECOM_SERVICE_REVENUE' AS account_name,
        CAST(0 AS DECIMAL(18,2)) AS debit_amount,
        journal_amount AS credit_amount,
        source_type,
        source_reference
    FROM journal_headers
    WHERE source_type = 'INVOICE'

    UNION ALL

    -- Payment: Debit Cash / Bank
    SELECT
        journal_id,
        posting_date,
        '1000' AS account_code,
        'CASH_BANK' AS account_name,
        journal_amount AS debit_amount,
        CAST(0 AS DECIMAL(18,2)) AS credit_amount,
        source_type,
        source_reference
    FROM journal_headers
    WHERE source_type = 'PAYMENT'

    UNION ALL

    -- Payment: Credit Accounts Receivable
    SELECT
        journal_id,
        posting_date,
        '1100' AS account_code,
        'ACCOUNTS_RECEIVABLE' AS account_name,
        CAST(0 AS DECIMAL(18,2)) AS debit_amount,
        journal_amount AS credit_amount,
        source_type,
        source_reference
    FROM journal_headers
    WHERE source_type = 'PAYMENT'
)

SELECT
    CONCAT(
        'GL',
        LPAD(
            CAST(ROW_NUMBER() OVER (
                ORDER BY journal_id, account_code
            ) AS STRING),
            10,
            '0'
        )
    ) AS gl_entry_id,

    journal_id,
    posting_date,
    account_code,
    account_name,
    CAST(debit_amount AS DECIMAL(18,2)) AS debit_amount,
    CAST(credit_amount AS DECIMAL(18,2)) AS credit_amount,
    source_type,
    source_reference,
    'ERP' AS source_system,
    'BATCH_001' AS ingest_batch_id,
    CURRENT_TIMESTAMP() AS ingestion_timestamp

FROM journal_lines;